# 06 · Limpieza: ENCUESTA-POST-EVENTOS (PRESENCIAL Y VIRTUAL)_anonimizada

Encuesta nueva posterior a los eventos (oct–nov 2026). Una fila = una respuesta. Incluye la pregunta **NPS** (de 0 a 10, ¿recomendarías este espacio?).

- Entrada: `datos/originales/ENCUESTA-POST-EVENTOS (PRESENCIAL Y VIRTUAL)_anonimizada.xlsx` (hoja `Responses`)
- Salidas:
  - `datos/limpios/06_encuesta_post_eventos_limpio.csv`: una fila por respuesta.
  - `datos/limpios/06_temas_interes.csv`: una fila por respuesta y tema elegido (la pregunta 18 es de selección múltiple). Sirve para hacer en Looker Studio el gráfico de temas más pedidos.

Se usan los mismos nombres de columna que en el 03 y el 04 donde la información es la misma (`id_respuesta`, `fecha_inicio`, `id_persona`, `ciudad`…).

**Limitación:** la encuesta no dice a qué evento corresponde cada respuesta. Solo se sabe la fecha y si la persona participó presencial o virtual.

In [ ]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Carga
Como en el 03 y el 04, la fila 0 está vacía y los encabezados están en la fila 1 (`header=1`).

In [ ]:
df = pd.read_excel(ORIGINALES / "ENCUESTA-POST-EVENTOS (PRESENCIAL Y VIRTUAL)_anonimizada.xlsx",
                   sheet_name="Responses", header=1)
assert df.shape == (53, 54), df.shape
df.head(3)

## 2. Renombrar y eliminar columnas
Varias preguntas tienen casi el mismo texto (la 15 se repite para presencial y virtual), así que cada columna se identifica por su **posición** y se comprueba con un fragmento de su nombre. Si el archivo cambia de orden, el `assert` avisa.

Se eliminan:
- `Unnamed: 0` (consecutivo), IP, correo y celular (vacías por la anonimización).
- `Nombre del recopilador`: un solo valor.
- Preguntas 9, 10 y 11 (cómo afiliarse, canal y horario de contacto): nadie las respondió.
- Pregunta 18 en una sola columna de texto: repite lo mismo que las columnas por tema y no se puede separar bien, porque algunos temas tienen comas en el nombre (`Nómina, compensación y beneficios`).

In [ ]:
COLUMNAS = {
    1: ("ID_RESPUESTA", "id_respuesta"),
    2: ("Respuesta iniciada", "fecha_inicio"),
    3: ("Respuesta completada", "fecha_fin"),
    6: ("ID_PERSONA", "id_persona"),
    9: ("ID_EMPRESA", "id_empresa"),
    10: ("NIVEL_CARGO", "nivel_cargo"),
    11: ("CIUDAD", "ciudad"),
    12: ("¿Eres afiliado", "es_afiliado"),
    13: ("hacer parte de ACTHO", "interes_afiliarse"),
    17: ("¿Cómo participaste?", "modalidad"),
    18: ("qué tan satisfecho", "satisfaccion"),
    19: ("De 0 a 10", "nps"),
    31: ("más te gustó", "lo_que_mas_gusto"),
    32: ("deberíamos mejorar", "que_mejorar"),
    48: ("tratamiento de datos", "acepta_tratamiento_datos"),
    49: ("recibir novedades", "recibir_info"),
    50: ("patrocinadores", "comparte_datos_patrocinadores"),
    51: ("use mi imagen", "autoriza_imagen"),
    52: ("EMPRESA_EN_CRM", "empresa_en_crm"),
    53: ("METODO_ASIGNACION", "metodo_asignacion"),
}
# Pregunta 15: una columna por aspecto, separada en presencial (20-25) y virtual (26-30)
ASPECTOS_PRESENCIAL = {20: "cal_expositor", 21: "cal_contenido", 22: "cal_recepcion",
                       23: "cal_espacio", 24: "cal_alimentacion", 25: "cal_tiempo"}
ASPECTOS_VIRTUAL = {26: "cal_expositor", 27: "cal_contenido", 28: "cal_recepcion",
                    29: "cal_espacio", 30: "cal_tiempo"}
# Pregunta 18: una columna por tema (33-46)
TEMAS = list(range(33, 47))

originales = df.columns
for i, (fragmento, _) in COLUMNAS.items():
    assert fragmento in originales[i], (i, originales[i])
assert all("presenciales" in originales[i] for i in ASPECTOS_PRESENCIAL)
assert all("virtuales" in originales[i] for i in ASPECTOS_VIRTUAL)
assert all(originales[i].startswith("18.") for i in TEMAS)

crudo = df
df = crudo.iloc[:, list(COLUMNAS)].copy()
df.columns = [nuevo for _, nuevo in COLUMNAS.values()]
df.columns.tolist()

## 3. Fechas
Mismo formato en español que el 03 y el 04 (`oct. 21, 2026 09:22:34 a. m.`), con la misma función.

In [ ]:
MESES = {"ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06",
         "jul": "07", "ago": "08", "sep": "09", "sept": "09", "oct": "10", "nov": "11", "dic": "12"}


def fecha_es(serie):
    texto = (serie.str.replace("\xa0", " ")
                  .str.replace(r"^([a-z]+)\.", lambda m: MESES[m.group(1)], regex=True)
                  .str.replace("a. m.", "AM").str.replace("p. m.", "PM"))
    return pd.to_datetime(texto, format="%m %d, %Y %I:%M:%S %p")


df["fecha_inicio"] = fecha_es(df["fecha_inicio"])
df["fecha_fin"] = fecha_es(df["fecha_fin"])
df["fecha_evento"] = df["fecha_inicio"].dt.normalize()
df["anio"] = df["fecha_inicio"].dt.year
df["mes"] = df["fecha_inicio"].dt.month

assert df["fecha_inicio"].notna().all() and df["fecha_fin"].notna().all()
assert (df["fecha_fin"] >= df["fecha_inicio"]).all()
df["fecha_evento"].value_counts().sort_index()

## 4. Calificaciones de la pregunta 15
La encuesta tiene un bloque para presencial y otro para virtual, con los mismos aspectos. Se juntan en una sola columna por aspecto (`cal_expositor`, `cal_contenido`…), y la columna `modalidad` dice de cuál bloque viene.
Así en Looker se puede comparar, por ejemplo, `cal_expositor` entre presencial y virtual sin manejar dos columnas.
- `cal_alimentacion` solo existe en presencial.
- En la encuesta, el aspecto "Manejo del tiempo" presencial tiene por error la descripción de alimentación. Se toma como manejo del tiempo, que es lo que dice el título.

In [ ]:
presencial = crudo.iloc[:, list(ASPECTOS_PRESENCIAL)].set_axis(list(ASPECTOS_PRESENCIAL.values()), axis=1)
virtual = crudo.iloc[:, list(ASPECTOS_VIRTUAL)].set_axis(list(ASPECTOS_VIRTUAL.values()), axis=1)

ambos = presencial.drop(columns="cal_alimentacion").notna().any(axis=1) & virtual.notna().any(axis=1)
assert not ambos.any(), "Hay respuestas con calificaciones presenciales y virtuales a la vez"

CALIFICACIONES = list(ASPECTOS_PRESENCIAL.values())
df[CALIFICACIONES] = presencial.combine_first(virtual)[CALIFICACIONES].astype("Int64")
assert df[CALIFICACIONES + ["satisfaccion"]].stack().between(1, 5).all()
df["satisfaccion"] = df["satisfaccion"].astype("Int64")
df["cal_promedio"] = df[CALIFICACIONES].mean(axis=1).round(2)
df[CALIFICACIONES].describe().loc[["count", "mean"]]

## 5. NPS
La respuesta viene como texto: `10 (Promoter)`, `8 (Passive)`. Se separa en:
- `nps`: el número de 0 a 10.
- `nps_categoria`: `Promotor` (9–10), `Pasivo` (7–8) o `Detractor` (0–6). Se calcula desde el número, no desde el texto, para que la regla sea siempre la misma.

En Looker Studio: **NPS = % promotores − % detractores**.

In [ ]:
df["nps"] = df["nps"].str.extract(r"^(\d+)", expand=False).astype("Int64")
assert df["nps"].dropna().between(0, 10).all()

df["nps_categoria"] = pd.cut(df["nps"].astype(float), bins=[-1, 6, 8, 10],
                             labels=["Detractor", "Pasivo", "Promotor"]).astype("string")
assert (df["nps"].isna() == df["nps_categoria"].isna()).all()
df["nps_categoria"].value_counts(dropna=False)

## 6. Respuestas incompletas
12 personas solo llenaron sus datos (afiliación y empresa) y no evaluaron el evento: no tienen modalidad, satisfacción ni NPS.
**Se conservan**, porque la pregunta de afiliación sí la respondieron, y se marcan con `evaluo_evento = False`. En Looker se filtran con esa columna al calcular satisfacción o NPS.

In [ ]:
df["evaluo_evento"] = df[["modalidad", "satisfaccion", "nps"]].notna().any(axis=1)
assert (df["modalidad"].notna() == df["evaluo_evento"]).all()
df["modalidad"] = df["modalidad"].fillna("Sin dato")
df["evaluo_evento"].value_counts()

## 7. Duplicados
Misma regla que el 03 y el 04: misma persona, mismo día y mismas respuestas de evaluación.
Hay 1 caso: P-02552 envió dos veces el formulario vacío el 27 de octubre. Se conserva el primero.

Otras 3 personas aparecen dos veces, pero **no son duplicados**: respondieron en días distintos (21 y 27 de octubre), es decir, sobre eventos distintos.

In [ ]:
EVALUACION = ["modalidad", "satisfaccion", "nps"] + CALIFICACIONES
duplicada = df.duplicated(["id_persona", "fecha_evento"] + EVALUACION)
print("Eliminadas:", df.loc[duplicada, "id_respuesta"].tolist())

df = df[~duplicada].reset_index(drop=True)
crudo = crudo[~duplicada.values].reset_index(drop=True)
assert len(df) == 52, len(df)

## 8. Textos y categorías
- **Ciudad:** `Medellin` → `Medellín`, `Bogota` y `Bogotá, Bogotá D.C., Colombia` → `Bogotá`. Vacío → `Sin dato`.
- **Afiliado:** `Sí, soy afiliado` / `No soy afiliado` → `True` / `False`.
- **Recibir información:** `Si` → `Sí`, para que coincida con el 03 y el 04. Vacíos → `Sin respuesta`, igual que en el 03 y el 04.
- **Autorizaciones:** vacíos → `Sin respuesta`.
- **Texto libre (preguntas 16 y 17):** se vacían respuestas sin contenido (`-`, `I`, `Ninguna`).

In [ ]:
CIUDADES = {"bogotá": "Bogotá", "bogota": "Bogotá", "bogotá, bogotá d.c., colombia": "Bogotá",
            "medellín": "Medellín", "medellin": "Medellín",
            "barranquilla": "Barranquilla", "envigado": "Envigado", "cota": "Cota"}
clave = df["ciudad"].str.strip().str.lower()
sin_mapear = set(clave.dropna()) - set(CIUDADES)
assert not sin_mapear, sin_mapear
df["ciudad"] = clave.map(CIUDADES).fillna("Sin dato")

AFILIADO = {"Sí, soy afiliado": True, "No soy afiliado": False}
assert set(df["es_afiliado"]) <= set(AFILIADO), set(df["es_afiliado"])
df["es_afiliado"] = df["es_afiliado"].map(AFILIADO)

df["recibir_info"] = df["recibir_info"].replace({"Si": "Sí"})
for c in ["interes_afiliarse", "acepta_tratamiento_datos", "recibir_info",
          "comparte_datos_patrocinadores", "autoriza_imagen"]:
    df[c] = df[c].str.strip().fillna("Sin respuesta")

SIN_CONTENIDO = {"-", ".", "i", "ninguna", "ninguno", "no", "n/a", "na"}
for c in ["lo_que_mas_gusto", "que_mejorar"]:
    texto = df[c].str.strip()
    df[c] = texto.mask(texto.str.lower().isin(SIN_CONTENIDO))

df["ciudad"].value_counts()

## 9. Temas de interés (pregunta 18)
Es de selección múltiple: cada tema es una columna que tiene el nombre del tema si la persona lo marcó, o está vacía si no.
Se pasa a una tabla larga (una fila por respuesta y tema), que es lo que Looker Studio necesita para contar qué temas se piden más.
En "Otro, ¿cuál?" se guarda lo que la persona escribió (`Compensación`, `Retención en la fuente`).

In [ ]:
temas = (crudo.iloc[:, TEMAS]
         .assign(id_respuesta=df["id_respuesta"])
         .melt(id_vars="id_respuesta", value_name="tema")
         .dropna(subset=["tema"])[["id_respuesta", "tema"]])
temas["tema"] = temas["tema"].str.strip().str.replace(r"^Otro, ¿cuál\? - ", "Otro: ", regex=True)
temas = temas.sort_values(["id_respuesta", "tema"]).reset_index(drop=True)

assert temas["id_respuesta"].isin(df["id_respuesta"]).all()
df["n_temas_elegidos"] = df["id_respuesta"].map(temas["id_respuesta"].value_counts()).fillna(0).astype(int)
temas["tema"].value_counts()

## 10. Validaciones y columnas finales

In [ ]:
assert df["id_respuesta"].is_unique
assert df["id_empresa"].str.fullmatch(r"EMP-\d{4}").all()
assert df["id_persona"].str.fullmatch(r"P-\d{5}").all()
assert set(df["modalidad"]) <= {"Presencial", "Virtual", "Sin dato"}

df = df[["id_respuesta", "fecha_inicio", "fecha_fin", "fecha_evento", "anio", "mes",
         "modalidad", "evaluo_evento", "satisfaccion", "nps", "nps_categoria",
         *CALIFICACIONES, "cal_promedio", "lo_que_mas_gusto", "que_mejorar", "n_temas_elegidos",
         "es_afiliado", "interes_afiliarse", "recibir_info", "acepta_tratamiento_datos",
         "comparte_datos_patrocinadores", "autoriza_imagen",
         "id_persona", "id_empresa", "nivel_cargo", "ciudad", "empresa_en_crm", "metodo_asignacion"]]
df.dtypes

## 11. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes. En Looker se conectan los dos archivos y se unen por `id_respuesta`.

In [ ]:
salida = LIMPIOS / "06_encuesta_post_eventos_limpio.csv"
salida_temas = LIMPIOS / "06_temas_interes.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")
temas.to_csv(salida_temas, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (52, 33)
assert pd.read_csv(salida_temas).shape == (len(temas), 2)
salida.name, salida_temas.name